In [10]:
!pip install azure-cosmos

In [ ]:
from azure.cosmos import CosmosClient
URI = "YOUR_URI"
KEY = "YOUR_KEY"

client = CosmosClient(URI, credential=KEY)

print("Connected successfully")

Connected successfully


In [12]:
from azure.cosmos import CosmosClient

client = CosmosClient(URI, credential=KEY)

database = client.get_database_client("MoviesDB")
container = database.get_container_client("movies")

print(container.read())

{'id': 'movies', 'indexingPolicy': {'indexingMode': 'consistent', 'automatic': True, 'includedPaths': [{'path': '/*'}], 'excludedPaths': [{'path': '/"_etag"/?'}], 'fullTextIndexes': []}, 'partitionKey': {'paths': ['/genre'], 'kind': 'Hash', 'version': 2}, 'uniqueKeyPolicy': {'uniqueKeys': []}, 'conflictResolutionPolicy': {'mode': 'LastWriterWins', 'conflictResolutionPath': '/_ts', 'conflictResolutionProcedure': ''}, 'geospatialConfig': {'type': 'Geography'}, 'fullTextPolicy': {'defaultLanguage': 'en-US', 'defaultSpec': {'language': 'en-US', 'stopWordListKind': 'extended'}, 'fullTextPaths': []}, '_rid': 'q7pLAIG6s-M=', '_ts': 1776082532, '_self': 'dbs/q7pLAA==/colls/q7pLAIG6s-M=/', '_etag': '"00005102-0000-4700-0000-69dcde640000"', '_docs': 'docs/', '_sprocs': 'sprocs/', '_triggers': 'triggers/', '_udfs': 'udfs/', '_conflicts': 'conflicts/', 'computedProperties': []}


In [16]:
movies = [
    {"id": "1", "title": "The Pursuit of Happyness", "genre": "Drama"},
    {"id": "2", "title": "Rain Man", "genre": "Drama"},
    {"id": "3", "title": "The Intern", "genre": "Drama"},
    {"id": "4", "title": "Coach Carter", "genre": "Sports"},
    {"id": "5", "title": "Fatherhood", "genre": "Drama"},
    {"id": "6", "title": "Forrest Gump", "genre": "Drama"},
    {"id": "7", "title": "Good Will Hunting", "genre": "Drama"},
    {"id": "8", "title": "A Beautiful Mind", "genre": "Biography"},
    {"id": "9", "title": "The Social Network", "genre": "Biography"},
    {"id": "10", "title": "The Shawshank Redemption", "genre": "Drama"},
    {"id": "11", "title": "Dead Poets Society", "genre": "Drama"},
    {"id": "12", "title": "The Blind Side", "genre": "Sports"},
    {"id": "13", "title": "Moneyball", "genre": "Sports"},
    {"id": "14", "title": "The Founder", "genre": "Biography"},
    {"id": "15", "title": "Jobs", "genre": "Biography"},
    {"id": "16", "title": "Steve Jobs", "genre": "Biography"},
    {"id": "17", "title": "Hidden Figures", "genre": "Drama"},
    {"id": "18", "title": "The Imitation Game", "genre": "Biography"},
    {"id": "19", "title": "Slumdog Millionaire", "genre": "Drama"},
    {"id": "20", "title": "Life of Pi", "genre": "Adventure"},
    {"id": "21", "title": "The Terminal", "genre": "Drama"},
    {"id": "22", "title": "Catch Me If You Can", "genre": "Biography"},
    {"id": "23", "title": "The Great Debaters", "genre": "Drama"},
    {"id": "24", "title": "Remember the Titans", "genre": "Sports"},
    {"id": "25", "title": "Rocky", "genre": "Sports"}
]

for movie in movies:
    try:
        response = container.create_item(movie)
        print("Inserted:", response["id"])
    except Exception as e:
        print("Error inserting:", movie["title"])
        print(e)

Inserted: 1
Inserted: 2
Inserted: 3
Inserted: 4
Inserted: 5
Inserted: 6
Inserted: 7
Inserted: 8
Inserted: 9
Inserted: 10
Inserted: 11
Inserted: 12
Inserted: 13
Inserted: 14
Inserted: 15
Inserted: 16
Inserted: 17
Inserted: 18
Inserted: 19
Inserted: 20
Inserted: 21
Inserted: 22
Inserted: 23
Inserted: 24
Inserted: 25


In [17]:
items = list(container.query_items(
    query="SELECT * FROM c",
    enable_cross_partition_query=True
))

print("Total items:", len(items))

for item in items:
    print(item["id"], "-", item["title"], "-", item["genre"])

Total items: 25
1 - The Pursuit of Happyness - Drama
2 - Rain Man - Drama
3 - The Intern - Drama
4 - Coach Carter - Sports
5 - Fatherhood - Drama
6 - Forrest Gump - Drama
7 - Good Will Hunting - Drama
8 - A Beautiful Mind - Biography
9 - The Social Network - Biography
10 - The Shawshank Redemption - Drama
11 - Dead Poets Society - Drama
12 - The Blind Side - Sports
13 - Moneyball - Sports
14 - The Founder - Biography
15 - Jobs - Biography
16 - Steve Jobs - Biography
17 - Hidden Figures - Drama
18 - The Imitation Game - Biography
19 - Slumdog Millionaire - Drama
20 - Life of Pi - Adventure
21 - The Terminal - Drama
22 - Catch Me If You Can - Biography
23 - The Great Debaters - Drama
24 - Remember the Titans - Sports
25 - Rocky - Sports


In [18]:
import random

items = list(container.query_items(
    query="SELECT * FROM c",
    enable_cross_partition_query=True
))

print("Found:", len(items), "movies")


for item in items:
    item["reviews"] = [
        {
            "userId": f"user{random.randint(1,5)}",
            "rating": random.randint(3,5),
            "comment": "Good movie"
        },
        {
            "userId": f"user{random.randint(1,5)}",
            "rating": random.randint(3,5),
            "comment": "Nice"
        }
    ]


    container.upsert_item(item)

print("All 25 movies updated successfully")

Found: 25 movies
All 25 movies updated successfully


In [19]:
items = list(container.query_items(
    query="SELECT * FROM c WHERE c.genre = 'Drama'",
    enable_cross_partition_query=True
))

In [21]:
items = list(container.query_items(
    query="SELECT * FROM c ORDER BY c.title",
    enable_cross_partition_query=True
))

In [22]:
items = list(container.query_items(
    query="SELECT VALUE COUNT(1) FROM c",
    enable_cross_partition_query=True
))
print(items)

[25]


## Azure Cosmos DB — Design Decisions

---

### Access Patterns

These are the core operations the database needs to support:

- **Get a movie with all its reviews** — the most common read, needs to be fast
- **Get movies by genre** — users browse and filter by category
- **Get reviews with high ratings** — surfaces top-rated content
- **Add reviews to movies** — the main write operation

---

### Design Decision — Embedding Reviews Inside Movies

Reviews are stored **embedded inside their parent movie document**, rather than in a separate collection.

This avoids joins and multiple queries — a single document fetch returns the movie *and* all its reviews. For a read-heavy workload like this, that's a significant win.

---

### Partition Key — `/genre`

`/genre` was chosen as the partition key because:

- Queries almost always filter by genre
- It gives Cosmos DB a meaningful way to route requests directly to the right partition
- It distributes data evenly enough to avoid hot partitions

---

### Trade-offs

|  Faster reads | One document = one query. Low latency, fewer RUs consumed. |
|  Heavier writes | Adding a review updates the whole movie document — worth monitoring as reviews grow. |

---

### Scaling

Data is distributed across partitions by genre, which prevents any single partition from becoming a bottleneck. As the catalogue grows, Cosmos handles the distribution automatically behind the scenes.
